# D3 · 99 · Comparison of the Dataset 3 experiments (saved outputs only)

Subject-level results on Dataset 3 (ds004504, 36 AD / 29 HC, within-D3 cross-validation): EARM-GNN and its two
ablations, the multi-band GCN, source and sensor PSD models and the age-only baseline. Tables, ROC curves,
confusion matrices and per-repeat paired differences (descriptive; no significance test). Nothing is trained.
Run after the D3 experiment notebooks; missing experiments are simply absent.
**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

In [ ]:
# ===== Dataset 3 setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_D3_CODE = False   # leave False. True only if you were asked to switch D3 to newer code
# Dataset 3 has its own code pin, <outputs>/D3_CODE_COMMIT.txt (all D3 notebooks run that commit);
# the D1/D2 pin CODE_COMMIT.txt is only reported, never read for checkout or changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'D3']
     + (['--update-code'] if UPDATE_D3_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])

## 1. Comparison

In [ ]:
sh("python scripts/exp7_earm.py compare --family D3")
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
cmp = sorted((Path(OUT) / 'experiments_D3' / '_comparison').glob('comparison_*'))[-1]
display(pd.read_csv(cmp / 'auc_by_setting.csv', index_col=0).round(3))
t = pd.read_csv(cmp / 'comparison_table.csv') if (cmp / 'comparison_table.csv').stat().st_size > 1 else pd.DataFrame()
if len(t):
    display(t[['model', 'connectivity', 'n_repeats', 'roc_auc_mean', 'roc_auc_sd', 'roc_auc_ci95_low',
               'roc_auc_ci95_high', 'balanced_accuracy_mean', 'sensitivity_mean', 'specificity_mean', 'f1_mean',
               'n_parameters', 'fit_seconds_mean']].round(3))
for png in sorted((cmp / 'figures').glob('D3_*.png')):
    print(png.name)
    display(Image(filename=str(png)))